# Cleaning & Exploring the June 2026 Unemployment Rate Data (by Metropolitan Statistical Area)

**Goal:** take the raw CSV, fix the usual data-quality problems, document every change,
and produce a clean dataset plus a few charts.

**Steps**

1. Load and inspect the raw file
2. Standardize column names and text values
3. Convert numeric columns
4. Remove duplicate rows
5. Report missing values
6. Explore the cleaned data (summary statistics and charts)
7. Save the cleaned dataset

**Tools:** Python, pandas, matplotlib

> To run this notebook, place the CSV in the `data/` folder (see the README).

## 1. Setup and loading

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("data") / "2026 June Unemployment Rate by Metropolitan Statistical Area (Percent).csv"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Text values that really mean "missing"
NULL_MARKERS = ["", "NA", "N/A", "n/a", "null", "NULL", "-"]

# Chart color (single series -> one color)
BAR_COLOR = "#2a78d6"

In [ ]:
raw = pd.read_csv(DATA_PATH)
df = raw.copy()

print(f"Rows: {df.shape[0]:,}   Columns: {df.shape[1]}")
df.head()

In [ ]:
df.info()

## 2. Standardize column names and text values

- Trim spaces and collapse repeated whitespace in column names
- Trim spaces in text cells
- Turn placeholder values such as `N/A`, `null` or `-` into real missing values

In [ ]:
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

text_cols = [
    col for col in df.columns
    if pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_object_dtype(df[col])
]

for col in text_cols:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace(NULL_MARKERS, pd.NA)

print("Text columns:", text_cols)

## 3. Convert numeric columns

A text column is converted to numbers when at least 80% of its non-missing values
can be parsed after removing thousands separators (`,`) and percent signs (`%`).
Values that cannot be parsed become missing and are counted in the report below.

In [ ]:
converted = []

for col in text_cols:
    cleaned = (
        df[col]
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
    )
    numeric = pd.to_numeric(cleaned, errors="coerce")

    non_missing = df[col].notna().sum()
    if non_missing > 0 and numeric.notna().sum() / non_missing >= 0.8:
        df[col] = numeric
        converted.append(col)

print("Converted to numeric:", converted)
print()
print(df.dtypes)

## 4. Remove duplicate rows

In [ ]:
duplicate_count = int(df.duplicated().sum())
print(f"Duplicate rows found: {duplicate_count}")

df = df.drop_duplicates().reset_index(drop=True)
print(f"Rows after removing duplicates: {len(df):,}")

## 5. Missing values

Missing values are **reported, not filled in**. Replacing gaps in unemployment rates with
a median or most-common value would invent data, so they stay empty. Rows where every
numeric column is empty carry no information and are dropped.

In [ ]:
missing_report = pd.DataFrame({
    "missing": df.isna().sum(),
    "percent": (df.isna().mean() * 100).round(1),
})
missing_report

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()

if num_cols:
    empty_rows = df[num_cols].isna().all(axis=1)
    print(f"Rows with no numeric values (dropped): {int(empty_rows.sum())}")
    df = df.loc[~empty_rows].reset_index(drop=True)

print(f"Final row count: {len(df):,}")

## 6. Explore the cleaned data

In [ ]:
df.describe().T

The charts use the **last numeric column** as the value to plot and the **first text column**
as the area name. If your file is laid out differently, change `VALUE_COL` and `LABEL_COL`.

In [ ]:
text_cols_left = df.select_dtypes(exclude="number").columns.tolist()

LABEL_COL = text_cols_left[0] if text_cols_left else None
VALUE_COL = num_cols[-1] if num_cols else None

print("Label column:", LABEL_COL)
print("Value column:", VALUE_COL)

In [ ]:
if VALUE_COL is not None:
    values = df[VALUE_COL].dropna()

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(values, bins=20, color=BAR_COLOR, edgecolor="white")
    ax.set_ylim(top=ax.get_ylim()[1] * 1.15)
    ax.axvline(values.median(), color="#52514e", linestyle="--", linewidth=1.5)
    ax.text(
        values.median(), ax.get_ylim()[1] * 0.97,
        f"  median {values.median():.1f}",
        color="#52514e", va="top",
    )

    ax.set_title(f"Distribution of {VALUE_COL} across {len(values):,} areas")
    ax.set_xlabel(VALUE_COL)
    ax.set_ylabel("Number of areas")
    ax.spines[["top", "right"]].set_visible(False)

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "distribution.png", dpi=150)
    plt.show()

In [ ]:
def plot_extremes(data, label_col, value_col, highest, filename):
    """Horizontal bar chart of the 10 highest or lowest areas."""
    subset = data[[label_col, value_col]].dropna()
    subset = subset.nlargest(10, value_col) if highest else subset.nsmallest(10, value_col)
    subset = subset.iloc[::-1]  # largest bar on top

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.barh(subset[label_col], subset[value_col], color=BAR_COLOR)
    ax.bar_label(bars, fmt="%.1f", padding=3)

    word = "Highest" if highest else "Lowest"
    ax.set_title(f"{word} 10 areas by {value_col}")
    ax.set_xlabel(value_col)
    ax.spines[["top", "right"]].set_visible(False)

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, dpi=150)
    plt.show()


if LABEL_COL is not None and VALUE_COL is not None:
    plot_extremes(df, LABEL_COL, VALUE_COL, highest=True, filename="top10.png")
    plot_extremes(df, LABEL_COL, VALUE_COL, highest=False, filename="bottom10.png")

## 7. Save the cleaned dataset

In [ ]:
cleaned_path = OUTPUT_DIR / "unemployment_cleaned.csv"
df.to_csv(cleaned_path, index=False)

print("Saved:", cleaned_path)
print(f"Rows before: {len(raw):,}  |  rows after: {len(df):,}")

## Summary of changes

| Step | What was done |
|---|---|
| Column names | Trimmed spaces, collapsed repeated whitespace |
| Text values | Trimmed spaces, placeholders (`N/A`, `null`, `-`) turned into missing values |
| Numbers | Text columns that are mostly numeric converted to numbers |
| Duplicates | Exact duplicate rows removed |
| Missing values | Reported, not filled; rows with no numeric data dropped |

The numbers for each step are printed in the cells above when the notebook is run.